## Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [15]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

True

In [16]:
model = init_chat_model(
    "groq:qwen/qwen3-32b",
    temperature=0.7,
)

In [17]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    plot: str = Field(..., description="The plot of the movie")
    genre: str = Field(..., description="The genre of the movie")
    rating: float = Field(..., description="The rating of the movie")

model_with_structured_output = model.with_structured_output(Movie)
model_with_structured_output

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11a4c96d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11b36b050>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'plot': {'description': 'The plot of the movie', 'type': 'string'}, 'genre': {'description': 'The genre of the movie', 'type': 'string'}, 'ra

In [18]:
response = model_with_structured_output.invoke(
    "Give me information about the movie 'The Matrix'"
)

In [19]:
response

Movie(title='The Matrix', year=1999, plot='A computer programmer discovers that his world is a simulated reality controlled by machines, and he must choose between continuing his normal life or joining a rebellion to save humanity.', genre='Science Fiction', rating=8.7)

#### Including raw output from the model

In [20]:

model_with_structured_output_v2 = model.with_structured_output(Movie, include_raw=True)
model_with_structured_output_v2

{
  raw: RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11a4c96d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11b36b050>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'plot': {'description': 'The plot of the movie', 'type': 'string'}, 'genre': {'description': 'The genre of the movie', 'type': 'stri

In [21]:
response = model_with_structured_output_v2.invoke(
    "Give me information about the movie 'The Matrix'"
)

In [22]:
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for information about the movie 'The Matrix'. Let me check what tools I have available. There's a Movie function that requires title, year, plot, genre, and rating. I need to fill in those parameters. First, I know the title is 'The Matrix' and the year it was released was 1999. The plot involves a computer programmer discovering a virtual reality. The genre is sci-fi, and the rating is probably high, like 8.7 on IMDb. Let me make sure all required fields are included. Yep, that's all covered. I'll structure the JSON with those details.\n", 'tool_calls': [{'id': 'aed7vs0rr', 'function': {'arguments': '{"genre":"Science Fiction","plot":"A computer programmer discovers that his life is a simulated reality and joins a group of rebels to fight against the machines that control it.","rating":8.7,"title":"The Matrix","year":1999}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token

#### Nested Structure

In [23]:
class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor")
    role: str = Field(..., description="The role played by the actor in the movie")
    character: str = Field(..., description="The character played by the actor in the movie")

In [29]:
class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    plot: str = Field(..., description="The plot of the movie")
    genre: list[str] = Field(..., description="All the genre of the movie")
    rating: float = Field(..., description="The rating of the movie")
    actors: list[Actor] = Field(..., description="The actors in the movie")

In [30]:
model_with_structured_output = model.with_structured_output(Movie)
model_with_structured_output

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11a4c96d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11b36b050>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'plot': {'description': 'The plot of the movie', 'type': 'string'}, 'genre': {'description': 'All the genre of the movie', 'items': {'type': 

In [31]:
response = model_with_structured_output.invoke(
    "Give me information about the movie 'The Matrix'"
)

In [32]:
print(response)

title='The Matrix' year=1999 plot='A computer programmer discovers a virtual reality and must fight against the system to save humanity.' genre=['Science Fiction', 'Action'] rating=8.7 actors=[Actor(name='Keanu Reeves', role='Lead', character='Neo'), Actor(name='Laurence Fishburne', role='Supporting', character='Morpheus'), Actor(name='Carrie-Anne Moss', role='Supporting', character='Trinity')]
